<h1>Approaches to Machine learning</h1>
<h2>Task 2</h2>
<p>Continuing from task 1 above, if required, find dataset of customer groceries shopping basket and build a recommended extra item model. For example, provided a shopping basket containing "pasta" and "olive oil", the model may make a recommendation of "canned tomato" as an extra item to be added to the shopping basket.

The outcome of the previous task should be useful for directly feeding into this task, you should look into reusing the output of the previous task for this task.

I would need to reduce each recipe down to only the pure ingredients, without any measurements. I then ask the user what food would they like? Next, I would want to train a markov chain on each recipe, ensuring it's are aware of usual food combinations by counting how often items like apple appears with cinnamon, and then display the results that pop up from the user inputted food. </p>

In [4]:
import ast
from collections import defaultdict
from nltk.tokenize import word_tokenize
import numpy as np
import pandas as pd
from sklearn.preprocessing import OneHotEncoder


In [5]:
recipe_df = pd.read_csv("./13k-recipes.csv", index_col=0, skiprows=0)
#Single out ingredients
recipe_df = recipe_df["Cleaned_Ingredients"]
#Getting rid of the first index as it wasn't following the correct column structure
recipe_df = recipe_df.drop(recipe_df.index[0])
#Turning the ingredients into a list as the whole thing was originally strings
recipe_df = recipe_df.apply(ast.literal_eval)

recipe_df[0]

C:\Users\Elune\AppData\Local\Temp\ipykernel_13908\3786275855.py:9: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  recipe_df[0]


['2 large egg whites',
 '1 pound new potatoes (about 1 inch in diameter)',
 '2 teaspoons kosher salt',
 '¾ teaspoon finely ground black pepper',
 '1 teaspoon finely chopped rosemary',
 '1 teaspoon finely chopped thyme',
 '1 teaspoon finely chopped parsley']

In [6]:
#for each ingredient, I want to make it into a list ["large", "egg", "white"] ["potatoes"...] and also have a list encompassing all of the ingredients in a recipe. I want a 3 lists, the overall list with everything in it, a list encompassing all of the recipes, and then a list encompassing all the ingredients in the recipe

#this list will have all of the data
all_recipes = []

for recipe in recipe_df:
    #this list will hold each ingredient
    ingredient_lists = []
    for ingredient in recipe:  
        #tokenising the words
        tokens = ingredient.split()   
        #getting rid of digits
        tokens = [t for t in tokens if not t.isdigit()]
        ingredient_lists.append(tokens)
    all_recipes.append(ingredient_lists)
all_recipes[0]

[['large', 'egg', 'whites'],
 ['pound', 'new', 'potatoes', '(about', 'inch', 'in', 'diameter)'],
 ['teaspoons', 'kosher', 'salt'],
 ['¾', 'teaspoon', 'finely', 'ground', 'black', 'pepper'],
 ['teaspoon', 'finely', 'chopped', 'rosemary'],
 ['teaspoon', 'finely', 'chopped', 'thyme'],
 ['teaspoon', 'finely', 'chopped', 'parsley']]

In [7]:
#all of the descriptor words that will be cleaned from the recipes
descriptors = "ground","cultured","gluten free","low fat","low calorie","vegan","organic", "low soidum", "boneless", "low sugar", "smoked", "locally grown", "chopped", "sliced", "diced", "fresh","freshly", "perserved", "powder", "kosher", "extra-sharp", "coarsley", "grated", "full-fat", "unsalted", "lightly", "beaten", "divided", "large", "small", "medium", "or", "cup", "ounces", "baking dish", "ceramic", "teaspoon", "storebought", "homemade", "Garnish", "twist", "Tbsp", "pat", "to taste","depending", "sweetness", "on", "the", "your", "optionally", "plus", "wheels", "dehydrated", "assorted", "such", "as", "other", "bone-in", "plus", "more", "piece", "thinly", "finely", "fine", "tsp", "stems", "stem", "removed", "packed", "serving", "for", "cut", "into", "wedges", "basic", "like", "Amy's", "fnishing", "can", "minced", "at", "room", "temperature", "hickory-smoked", "uncured", "fully" "coooked", "peeled", "halved", "(such","Persian", "English","hothouse,", "2½ ","lengthwise", "divided,",  "unsweetend", "smooth", "grainy", "if", "needed", "ml", "l", "fl", "oz", "ounce", "lb", "kg", "pound", "in", "diameter","ounces)", "taste", "teaspoons", "halves", "garnish", "about", "inch", "'", ",", ")", "(", "[", "]", ".", "'16", "to", "whole", "new","(2",  "1kg", "2kg", "tablespoons", "tablespoon", "(Japanese", "breadcrumbs)", "crushed"  "pure", "and'","140g", "1/4", "320g", "40g", "'10", "¾", "¼", "1–1½", "raw", "skinless", "cups", "and/or", "whole-milk","'1-pound)'",  "½", "g", "6–7","(¾-", "lb.", '4-qt', "dish", "glass", "shallow", "baking", "Garnish:", "and", "Tbsp.", '(about', "(optional)", 'Dehydrated', 'finishing', '1½', "tsp.", 'needed)', '(if', "torn", '(for', 'serving)', "with", "leaves","thin-skinned", "shredded", '(6g', 'oz)', 'diameter)',"1/2", "stick","coarsley","(8", "whites", "yolks" '(about',"oz.", "(use", "a", "mandoline", "you", "have", "one)", "(such", "(1", "stick)", "(65", "g)", "(packed;", "(250", "(to", "taste)", "(optional", "see", "Pre-Prep", "below)", "2-4", "salt", "black", "pepper",  #getting rid of salt and black pepper since theyre over represented


In [8]:
#applying the descriptor words
cleaned_recipes = []

for recipe in all_recipes:
    cleaned_recipe = []
    for ingredient in recipe:
        cleaned_ingredient = [
            token for token in ingredient
            if token.lower() not in descriptors
        ]
        #append all the non empty ingredients 
        if cleaned_ingredient:  
            cleaned_recipe.append(cleaned_ingredient)
    #append it all to the cleaned_recipes
    cleaned_recipes.append(cleaned_recipe)

cleaned_recipes[0]

[['egg'], ['potatoes'], ['rosemary'], ['thyme'], ['parsley']]

In [9]:
#joining the tokens such as "egg" and "whites" together to create one ingredient
cleaned_recipes_joined = []
for recipe in cleaned_recipes:
    joined_ingredients = []
    for ingredient in recipe:
        #joining it by adding a space inetween
        joined_ingredient = " ".join(ingredient)
        joined_ingredients.append(joined_ingredient)

    cleaned_recipes_joined.append(joined_ingredients)

cleaned_recipes_joined[0]


['egg', 'potatoes', 'rosemary', 'thyme', 'parsley']

In [14]:
#creating an empty transition dictionary that converts the value into an int
transitions = defaultdict(lambda: defaultdict(int))

for recipe in cleaned_recipes_joined:
    for ingredient in recipe:
        for other in recipe:
            #creation of the tracker
            if ingredient != other:
                transitions[ingredient][other] += 1



In [11]:
#Next we must convert the numbers into probabilities, I will be using the conditional probabilities formula for this which is : P(next∣current)=
#count(current → next) / ∑all outgoing counts from current
prob_transitions = {}

for ingred, next_items in transitions.items():
    #sum of outgoing transitions
    total = sum(next_items.values()) 
    
    #adding the ingredient title before each dictionary
    prob_transitions[ingred] = {}
    #normalising the data
    for next_ing, freq in next_items.items():
        prob_transitions[ingred][next_ing] = freq / total


In [12]:
enquiry = input(str("What food are you looking for? "))

reccomended = max(
    prob_transitions[enquiry],
    key=prob_transitions[enquiry].get
)
print(reccomended)

honey
